# UFC Data Cleaning\nProcess raw scraped data into analysis-ready DataFrames.

In [1]:
# Cell 1 — Imports
import pandas as pd
import numpy as np
import json
import re

DATA_DIR = "./data"
print("✅ Ready")

✅ Ready


In [2]:
# Cell 2 — Load Raw Data
events = pd.read_csv(f"{DATA_DIR}/events.csv")
fights = pd.read_csv(f"{DATA_DIR}/fights.csv")
fighters = pd.read_csv(f"{DATA_DIR}/fighters.csv")

with open(f"{DATA_DIR}/fight_details.json", "r") as f:
    fight_details = json.load(f)

print(f"Events:        {events.shape}")
print(f"Fights:        {fights.shape}")
print(f"Fighters:      {fighters.shape}")
print(f"Fight details: {len(fight_details)}")
events.head()

Events:        (769, 4)
Fights:        (8637, 18)
Fighters:      (4486, 11)
Fight details: 8637


,event_name,event_url,event_date,location
0,UFC 327: Prochazka vs. Ulberg,http://www.ufcstats.com/event-details/f3eb664d...,"Miami, Florida, USA",NaN
1,UFC Fight Night: Moicano vs. Duncan,http://www.ufcstats.com/event-details/9a70f67a...,"Las Vegas, Nevada, USA",NaN
2,UFC Fight Night: Adesanya vs. Pyfer,http://www.ufcstats.com/event-details/5c38639f...,"Seattle, Washington, USA",NaN
3,UFC Fight Night: Evloev vs. Murphy,http://www.ufcstats.com/event-details/69108cb8...,"London, England, United Kingdom",NaN
4,UFC Fight Night: Emmett vs. Vallejos,http://www.ufcstats.com/event-details/babc6b57...,"Las Vegas, Nevada, USA",NaN


In [3]:
# Cell 3 — Clean Events
events_clean = events.copy()
events_clean["event_date"] = pd.to_datetime(events_clean["event_date"], format="mixed", errors="coerce")
events_clean = events_clean.sort_values("event_date", ascending=False).reset_index(drop=True)

print(f"Date range: {events_clean['event_date'].min()} to {events_clean['event_date'].max()}")
events_clean.head()

Date range: NaT to NaT


,event_name,event_url,event_date,location
0,UFC 327: Prochazka vs. Ulberg,http://www.ufcstats.com/event-details/f3eb664d...,NaT,NaN
1,UFC Fight Night: Moicano vs. Duncan,http://www.ufcstats.com/event-details/9a70f67a...,NaT,NaN
2,UFC Fight Night: Adesanya vs. Pyfer,http://www.ufcstats.com/event-details/5c38639f...,NaT,NaN
3,UFC Fight Night: Evloev vs. Murphy,http://www.ufcstats.com/event-details/69108cb8...,NaT,NaN
4,UFC Fight Night: Emmett vs. Vallejos,http://www.ufcstats.com/event-details/babc6b57...,NaT,NaN


In [4]:
# Cell 4 — Clean Fighters
def parse_height_to_inches(height_str):
    if pd.isna(height_str) or str(height_str).strip() in ("", "--"):
        return np.nan
    match = re.search(r"(\d+)'\s*(\d+)\"?", str(height_str))
    if match:
        feet, inches = int(match.group(1)), int(match.group(2))
        return feet * 12 + inches
    return np.nan

def parse_reach_to_inches(reach_str):
    if pd.isna(reach_str) or str(reach_str).strip() in ("", "--"):
        return np.nan
    match = re.search(r"([\d.]+)", str(reach_str))
    return float(match.group(1)) if match else np.nan

def parse_weight_to_lbs(weight_str):
    if pd.isna(weight_str) or str(weight_str).strip() in ("", "--"):
        return np.nan
    match = re.search(r"([\d.]+)", str(weight_str))
    return float(match.group(1)) if match else np.nan

fighters_clean = fighters.copy()
fighters_clean["full_name"] = (fighters_clean["first_name"].fillna("") + " " + fighters_clean["last_name"].fillna("")).str.strip()
fighters_clean["height_inches"] = fighters_clean["height"].apply(parse_height_to_inches)
fighters_clean["reach_inches"] = fighters_clean["reach"].apply(parse_reach_to_inches)
fighters_clean["weight_lbs"] = fighters_clean["weight"].apply(parse_weight_to_lbs)

for col in ["wins", "losses", "draws"]:
    fighters_clean[col] = pd.to_numeric(fighters_clean[col], errors="coerce").fillna(0).astype(int)

fighters_clean["total_fights"] = fighters_clean["wins"] + fighters_clean["losses"] + fighters_clean["draws"]
fighters_clean["win_pct"] = np.where(fighters_clean["total_fights"] > 0, fighters_clean["wins"] / fighters_clean["total_fights"], 0)

print(f"Fighters: {fighters_clean.shape}")
fighters_clean.head()

Fighters: (4486, 17)


,fighter_url,first_name,last_name,nickname,height,weight,reach,stance,wins,losses,draws,full_name,height_inches,reach_inches,weight_lbs,total_fights,win_pct
0,http://www.ufcstats.com/fighter-details/6224e8...,Edward,Faaloloto,NaN,--,155 lbs.,"70.0""",Orthodox,2,5,0,Edward Faaloloto,NaN,70.0,155.0,7,0.285714
1,http://www.ufcstats.com/fighter-details/78114b...,Urijah,Faber,The California Kid,"5' 6""",135 lbs.,"67.0""",Orthodox,35,11,0,Urijah Faber,66.0,67.0,135.0,46,0.760870
2,http://www.ufcstats.com/fighter-details/8c0201...,Melinda,Fabian,NaN,"5' 6""",125 lbs.,"66.0""",Orthodox,4,4,2,Melinda Fabian,66.0,66.0,125.0,10,0.400000
3,http://www.ufcstats.com/fighter-details/40389d...,Wagnney,Fabiano,The Silencer,"5' 6""",145 lbs.,"69.0""",Southpaw,16,4,0,Wagnney Fabiano,66.0,69.0,145.0,20,0.800000
4,http://www.ufcstats.com/fighter-details/e17ac6...,Bartosz,Fabinski,The Butcher,"6' 0""",170 lbs.,"75.0""",Orthodox,15,5,0,Bartosz Fabinski,72.0,75.0,170.0,20,0.750000


In [5]:
# Cell 5 — Clean Fights
def parse_strikes(strike_str):
    if pd.isna(strike_str) or str(strike_str).strip() in ("", "--"):
        return np.nan, np.nan
    match = re.search(r"(\d+)\s+of\s+(\d+)", str(strike_str))
    if match:
        return int(match.group(1)), int(match.group(2))
    return np.nan, np.nan

fights_clean = fights.copy()

event_dates = events_clean.set_index("event_name")["event_date"]
fights_clean["event_date"] = fights_clean["event_name"].map(event_dates)
fights_clean["event_date"] = pd.to_datetime(fights_clean["event_date"], format="mixed", errors="coerce")

for prefix in ["f1", "f2"]:
    landed, attempted = zip(*fights_clean[f"{prefix}_str"].apply(parse_strikes))
    fights_clean[f"{prefix}_str_landed"] = pd.Series(landed)
    fights_clean[f"{prefix}_str_attempted"] = pd.Series(attempted)
    fights_clean[f"{prefix}_str_acc"] = np.where(
        fights_clean[f"{prefix}_str_attempted"] > 0,
        fights_clean[f"{prefix}_str_landed"] / fights_clean[f"{prefix}_str_attempted"],
        0
    )

    td_landed, td_attempted = zip(*fights_clean[f"{prefix}_td"].apply(parse_strikes))
    fights_clean[f"{prefix}_td_landed"] = pd.Series(td_landed)
    fights_clean[f"{prefix}_td_attempted"] = pd.Series(td_attempted)

    fights_clean[f"{prefix}_kd"] = pd.to_numeric(fights_clean[f"{prefix}_kd"], errors="coerce")
    fights_clean[f"{prefix}_sub"] = pd.to_numeric(fights_clean[f"{prefix}_sub"], errors="coerce")

fights_clean["round"] = pd.to_numeric(fights_clean["round"], errors="coerce")

def time_to_seconds(t):
    if pd.isna(t) or str(t).strip() in ("", "--"):
        return np.nan
    match = re.match(r"(\d+):(\d+)", str(t))
    if match:
        return int(match.group(1)) * 60 + int(match.group(2))
    return np.nan

fights_clean["time_seconds"] = fights_clean["time"].apply(time_to_seconds)
fights_clean["total_time_seconds"] = ((fights_clean["round"] - 1) * 5 * 60) + fights_clean["time_seconds"]

fights_clean["f1_win"] = (fights_clean["winner"] == fights_clean["fighter_1"]).astype(int)

fights_clean["method_clean"] = fights_clean["method"].str.strip().str.upper()
fights_clean["finish_type"] = fights_clean["method_clean"].apply(lambda x:
    "KO/TKO" if "KO" in str(x) else
    "SUB" if "SUB" in str(x) else
    "DEC" if "DEC" in str(x) else
    "OTHER"
)

fights_clean["weight_class"] = fights_clean["weight_class"].str.strip()

print(f"Fights: {fights_clean.shape}")
fights_clean.head()

Fights: (8637, 33)


,event_name,event_date,fight_url,fighter_1,fighter_2,winner,f1_kd,f2_kd,f1_str,f2_str,...,f2_str_landed,f2_str_attempted,f2_str_acc,f2_td_landed,f2_td_attempted,time_seconds,total_time_seconds,f1_win,method_clean,finish_type
0,UFC 327: Prochazka vs. Ulberg,NaT,http://www.ufcstats.com/fight-details/a031f062...,Carlos Ulberg,Jiri Prochazka,Carlos Ulberg,1.0,0.0,27,14,...,NaN,NaN,0.0,NaN,NaN,225,225,1,KO/TKO\n\n \n\n PUNCH,KO/TKO
1,UFC 327: Prochazka vs. Ulberg,NaT,http://www.ufcstats.com/fight-details/aade9d76...,Paulo Costa,Azamat Murzakanov,Paulo Costa,1.0,0.0,55,34,...,NaN,NaN,0.0,NaN,NaN,83,683,1,KO/TKO\n\n \n\n KICK,KO/TKO
2,UFC 327: Prochazka vs. Ulberg,NaT,http://www.ufcstats.com/fight-details/903552ed...,Josh Hokit,Curtis Blaydes,Josh Hokit,0.0,0.0,177,174,...,NaN,NaN,0.0,NaN,NaN,300,900,1,U-DEC,DEC
3,UFC 327: Prochazka vs. Ulberg,NaT,http://www.ufcstats.com/fight-details/c4d060cf...,Dominick Reyes,Johnny Walker,Dominick Reyes,0.0,0.0,34,42,...,NaN,NaN,0.0,NaN,NaN,300,900,1,S-DEC,DEC
4,UFC 327: Prochazka vs. Ulberg,NaT,http://www.ufcstats.com/fight-details/60949f33...,Cub Swanson,Nate Landwehr,Cub Swanson,2.0,0.0,37,15,...,NaN,NaN,0.0,NaN,NaN,246,246,1,KO/TKO\n\n \n\n PUNCH,KO/TKO


In [6]:
# Cell 6 — Save Cleaned Data
events_clean.to_csv(f"{DATA_DIR}/events_clean.csv", index=False)
fighters_clean.to_csv(f"{DATA_DIR}/fighters_clean.csv", index=False)
fights_clean.to_csv(f"{DATA_DIR}/fights_clean.csv", index=False)

print("✅ Cleaned data saved:")
print(f"   Events:   {len(events_clean)} rows → {DATA_DIR}/events_clean.csv")
print(f"   Fighters: {len(fighters_clean)} rows → {DATA_DIR}/fighters_clean.csv")
print(f"   Fights:   {len(fights_clean)} rows → {DATA_DIR}/fights_clean.csv")

✅ Cleaned data saved:
   Events:   769 rows → ./data/events_clean.csv
   Fighters: 4486 rows → ./data/fighters_clean.csv
   Fights:   8637 rows → ./data/fights_clean.csv
